# Time Series Forecasting — AutoRegressive (AR) Model


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.ar_model import AutoReg  # AR is deprecated; use AutoReg only
from statsmodels.tsa.stattools import adfuller

In [ ]:
df = pd.read_csv('daily-min-temperatures.csv', parse_dates=['Date'], index_col='Date')
print(df.head())

In [ ]:
# Plot the time series
plt.figure(figsize=(10, 4))
plt.plot(df['Temp'])
plt.title('Daily Minimum Temperatures')
plt.xlabel('Date')
plt.ylabel('Temperature')
plt.tight_layout()
plt.show()

## Check For Stationarity (ADF Test)


In [ ]:
# FIX: Removed erroneous indentation that would cause IndentationError
dftest = adfuller(df['Temp'], autolag='AIC')
print('1. ADF : ', dftest[0])
print('2. P-Value : ', dftest[1])
print('3. Num Of Lags : ', dftest[2])
print('4. Num Of Observations Used For ADF Regression and Critical Values Calculation :', dftest[3])
print('5. Critical Values :')
for key, val in dftest[4].items():
    print('\t', key, ': ', val)

## ACF and PACF Plots
Use these to determine the appropriate number of lags for the AR model.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_pacf(df['Temp'], lags=25, ax=axes[0])
axes[0].set_title('Partial Autocorrelation (PACF)')
plot_acf(df['Temp'], lags=25, ax=axes[1])
axes[1].set_title('Autocorrelation (ACF)')
plt.tight_layout()
plt.show()

## Train-Test Split


In [ ]:
# Hold out last 7 days for testing
train = df['Temp'][:len(df['Temp']) - 7]
test  = df['Temp'][len(df['Temp']) - 7:]
print(f'Training size : {len(train)}')
print(f'Test size     : {len(test)}')

## Fit AutoReg Model


In [ ]:
model = AutoReg(train, lags=20)
model_fit = model.fit()
print(model_fit.summary())

## Make Predictions on Test Set


In [ ]:
# dynamic=False: uses actual historical values for prediction (correct for evaluation)
pred = model_fit.predict(
    start=len(train),
    end=len(train) + len(test) - 1,
    dynamic=False
)
# Assign the test datetime index to predictions for clean plotting
pred.index = test.index

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(test,       label='Actual',    color='blue')
plt.plot(pred,       label='Predicted', color='red', linestyle='--')
plt.title('AR Model — Test Set: Predicted vs Actual')
plt.xlabel('Date')
plt.ylabel('Temperature')
plt.legend()
plt.tight_layout()
plt.show()

## Calculate RMSE


In [ ]:
from math import sqrt
from sklearn.metrics import mean_squared_error

rmse = sqrt(mean_squared_error(test, pred))
print(f'RMSE: {rmse:.3f}')

## Future Forecast (Next 7 Days)


In [ ]:
# FIX: Use dynamic=True for genuine future forecasting beyond observed data.
# Beyond the dataset there are no real values, so the model must use its own
# predictions recursively — dynamic=True is the honest setting here.
future_start = len(train) + len(test)
future_end   = future_start + 6  # 7 days ahead

future_pred = model_fit.predict(start=future_start, end=future_end, dynamic=True)

# Build a proper date index for the future window
last_date    = df.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=7, freq='D')
future_pred.index = future_dates

print('Future Predictions (next 7 days):')
print(future_pred)

In [ ]:
# Plot the last 30 days of actual data + 7-day future forecast
plt.figure(figsize=(10, 4))
plt.plot(df['Temp'].iloc[-30:], label='Historical (last 30 days)', color='blue')
plt.plot(future_pred,           label='Future Forecast (7 days)',  color='green', linestyle='--', marker='o')
plt.title('AR Model — 7-Day Future Forecast')
plt.xlabel('Date')
plt.ylabel('Temperature')
plt.legend()
plt.tight_layout()
plt.show()